## Example 1: StateGraph construction pattern

In [ ]:
from typing import TypedDict, List, Dict
from dataclasses import dataclass

# Example 1: StateGraph construction pattern
class GraphState(TypedDict):
    """Define the state structure for the graph"""
    messages: List[str]
    current_node: str
    metadata: Dict

class StateGraph:
    """Simple state graph implementation"""
    def __init__(self, state_class):
        self.state_class = state_class
        self.nodes = {}
        self.edges = {}
        self.entry_point = None
        
    def add_node(self, name: str, func: callable):
        """Add a node to the graph"""
        self.nodes[name] = func
        return self
        
    def set_entry_point(self, node: str):
        """Set the starting node"""
        self.entry_point = node

# Create a simple workflow graph
graph = StateGraph(GraphState)
graph.add_node("start", lambda x: {"messages": ["Starting"]})
graph.add_node("process", lambda x: {"messages": x["messages"] + ["Processing"]})
graph.set_entry_point("start")

print(f"Graph created with {len(graph.nodes)} nodes")
print(f"Entry point: {graph.entry_point}")

## Example 2: Conditional edges and routing

In [ ]:
from typing import Union, Literal

# Example 2: Conditional edges and routing
class ConditionalRouter:
    """Implement conditional routing in graphs"""
    
    def __init__(self):
        self.conditions = {}
        
    def add_conditional_edge(self, source: str, condition: callable, routes: dict):
        """Add conditional routing from a node"""
        self.conditions[source] = {
            "condition": condition,
            "routes": routes
        }
        
    def route(self, source: str, state: dict) -> str:
        """Determine next node based on condition"""
        if source in self.conditions:
            result = self.conditions[source]["condition"](state)
            return self.conditions[source]["routes"].get(result, "end")
        return "end"

# Setup conditional routing
router = ConditionalRouter()
router.add_conditional_edge(
    "decision",
    lambda x: "continue" if len(x.get("items", [])) > 0 else "stop",
    {"continue": "process", "stop": "end"}
)

# Test routing
state1 = {"items": [1, 2, 3]}
state2 = {"items": []}
print(f"Route with items: {router.route('decision', state1)}")
print(f"Route without items: {router.route('decision', state2)}")

## Example 3: Parallel node execution

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import time

# Example 3: Parallel node execution
class ParallelExecutor:
    """Execute graph nodes in parallel"""
    
    def __init__(self, max_workers: int = 3):
        self.executor = ThreadPoolExecutor(max_workers=max_workers)
        
    def execute_parallel(self, nodes: list, state: dict) -> list:
        """Execute multiple nodes in parallel"""
        def run_node(node_func):
            # Simulate some processing
            result = node_func(state)
            return result
        
        futures = [self.executor.submit(run_node, func) for func in nodes]
        results = [f.result() for f in futures]
        return results
    
    def shutdown(self):
        self.executor.shutdown()

# Define parallel tasks
tasks = [
    lambda x: {"task": "fetch", "data": len(x.get("input", ""))},
    lambda x: {"task": "validate", "valid": True},
    lambda x: {"task": "transform", "output": "processed"}
]

# Execute in parallel
executor = ParallelExecutor()
results = executor.execute_parallel(tasks, {"input": "test data"})
executor.shutdown()

print(f"Executed {len(tasks)} tasks in parallel")
for r in results:
    print(f"  - {r['task']}: {r}")

## Example 4: Message passing and channels

In [ ]:
from collections import defaultdict
from typing import Any

# Example 4: Message passing and channels
class ChannelManager:
    """Manage message passing between nodes via channels"""
    
    def __init__(self):
        self.channels = defaultdict(list)
        self.subscribers = defaultdict(list)
        
    def publish(self, channel: str, message: Any):
        """Publish message to a channel"""
        self.channels[channel].append(message)
        # Notify subscribers
        for callback in self.subscribers[channel]:
            callback(message)
            
    def subscribe(self, channel: str, callback: callable):
        """Subscribe to channel updates"""
        self.subscribers[channel].append(callback)
        
    def get_messages(self, channel: str) -> list:
        """Get all messages from a channel"""
        return self.channels[channel].copy()

# Setup channels
channels = ChannelManager()

# Subscribe to channels
channels.subscribe("events", lambda m: print(f"Event received: {m}"))

# Publish messages
channels.publish("events", {"type": "start", "node": "input"})
channels.publish("events", {"type": "process", "node": "transform"})
channels.publish("state", {"data": "processed"})

print(f"\nChannel 'state' messages: {channels.get_messages('state')}")

## Example 5: Graph compilation and execution

In [ ]:
# Example 5: Graph compilation and execution
class CompiledGraph:
    """Compile and execute a complete graph workflow"""
    
    def __init__(self, nodes: dict, edges: dict, entry: str):
        self.nodes = nodes
        self.edges = edges
        self.entry = entry
        self.execution_order = self._compile()
        
    def _compile(self) -> list:
        """Compile graph into execution order"""
        visited = set()
        order = []
        
        def visit(node):
            if node not in visited:
                visited.add(node)
                order.append(node)
        
        visit(self.entry)
        for src, dst in self.edges.items():
            visit(dst)
        return order
        
    def invoke(self, input_state: dict) -> dict:
        """Execute the compiled graph"""
        state = input_state.copy()
        for node in self.execution_order:
            if node in self.nodes:
                result = self.nodes[node](state)
                state.update(result)
        return state

# Build and compile a workflow
nodes = {
    "input": lambda s: {"step": "input", "value": s.get("initial", 0)},
    "process": lambda s: {"step": "process", "value": s["value"] * 2},
    "output": lambda s: {"step": "output", "result": s["value"]}
}
edges = {"input": "process", "process": "output"}

graph = CompiledGraph(nodes, edges, "input")
result = graph.invoke({"initial": 5})

print(f"Execution order: {graph.execution_order}")
print(f"Final result: {result}")